# Evening peak avoidance — "Avondpiek mijden"

Demo of the evening peak avoidance analysis, built for the *Piek Mijden Huishoudens Utrecht*
campaign: households in the Utrecht congestion area shift consumption out of the 16:00–21:00
window during November–February.

The analysis computes two quantities per day, from the quarter-hourly gross meter registers:

| | |
|---|---|
| **Avondpiek** | the highest quarter-hour power inside the window, in kW — how hard the connection loads the grid at its worst moment |
| **Piekaandeel** | the share of net daily offtake falling inside the window, in percent — how much of the day sits in the evening, which is what a participant can actually shift |

Injection is clipped to zero **per quarter-hour, before any summation**, so the share stays
between 0 and 100% and stays comparable between households with and without solar panels.

Spreading consumption across the day is good; pulling it all into one moment is not.

## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

from openenergyid.evening_peak import (
    EveningPeakAnalyzer,
    EveningPeakInput,
    EveningPeakOutput,
)

## 1. Load the sample input

One campaign winter for a single Dutch household, generated by
`data/evening_peak/make_sample.py`. It is assembled from appliance events rather than from an
average profile, so consecutive days differ the way real ones do: an oven at 19:00 draws 3–5 kW,
an evening where nobody is home barely more than the fridge. There is a deliberate three-day gap
in January, standing in for a break in the P4 feed.

In [3]:
sample = Path("data/evening_peak/evening_peak_sample.json")
analysis_input = EveningPeakInput.model_validate_json(sample.read_text(encoding="utf-8"))

print(f"timezone        : {analysis_input.timezone}")
print(f"window          : {analysis_input.window_start:%H:%M} – {analysis_input.window_end:%H:%M}")
print(f"threshold       : {analysis_input.peak_share_threshold:.0%}")
print(f"quarter-hours   : {len(analysis_input.gross_offtake.index):,}")
print(
    f"first / last    : {analysis_input.gross_offtake.first_timestamp()}"
    f" / {analysis_input.gross_offtake.last_timestamp()}"
)

timezone        : Europe/Amsterdam
window          : 16:00 – 21:00
threshold       : 37%
quarter-hours   : 11,232
first / last    : 2026-11-01 00:00:00+01:00 / 2027-02-28 23:45:00+01:00


## 2. Run the analysis

Two steps: `prepare_net_offtake` turns the two gross registers into one non-negative net series
in the connection's own timezone, and `analyze` reduces it to one row per local day plus the
weekly medians.

In [4]:
analyzer = EveningPeakAnalyzer(
    timezone=analysis_input.timezone,
    window_start=analysis_input.window_start,
    window_end=analysis_input.window_end,
    peak_share_threshold=analysis_input.peak_share_threshold,
    min_day_coverage=analysis_input.min_day_coverage,
)

gross_offtake, gross_injection = analysis_input.to_polars()
net_offtake = analyzer.prepare_net_offtake(gross_offtake, gross_injection)
result = analyzer.analyze(net_offtake)

daily = result.daily.collect()
week_medians = result.week_medians.collect()

daily.select(
    "day",
    "evening_peak_in_kilowatt",
    "evening_peak_share_in_percent",
    "daily_offtake_in_kilowatthour",
    "observed_quarters",
    "is_complete",
).head()

day,evening_peak_in_kilowatt,evening_peak_share_in_percent,daily_offtake_in_kilowatthour,observed_quarters,is_complete
"datetime[μs, Europe/Amsterdam]",f64,f64,f64,i64,bool
2026-11-01 00:00:00 CET,2.1564,24.161594,14.7035,96,true
2026-11-02 00:00:00 CET,1.6644,30.195308,10.0764,96,true
2026-11-03 00:00:00 CET,4.006,33.117651,12.5184,96,true
2026-11-04 00:00:00 CET,4.022,42.507136,15.7997,96,true
2026-11-05 00:00:00 CET,4.9464,47.315382,16.5517,96,true


### Chart styling

One hue at two steps: the daily series thin and translucent, the weekly median solid and stepped
over it. Same measure at two resolutions, so it is one colour rather than two — and the
lightness difference carries the distinction for colourblind readers as well as the hue would.
Grid and axes stay recessive.

In [5]:
SERIES = "#2a78d6"
SERIES_FAINT = "rgba(42, 120, 214, 0.45)"
SERIES_FILL = "rgba(42, 120, 214, 0.10)"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GRID = "rgba(11, 11, 11, 0.08)"
SURFACE = "#fcfcfb"


def style(figure: go.Figure, title: str, subtitle: str, unit: str) -> go.Figure:
    """Apply the shared chart frame."""
    figure.update_layout(
        title={
            "text": f"<b>{title}</b><br><span style='font-size:12px;color:{INK_MUTED}'>{subtitle}</span>",
            "font": {"size": 16, "color": INK},
            "x": 0,
            "xref": "paper",
        },
        template="simple_white",
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font={"family": "system-ui, sans-serif", "size": 12, "color": INK_MUTED},
        height=380,
        margin={"l": 60, "r": 30, "t": 90, "b": 50},
        hovermode="x unified",
        legend={"orientation": "h", "yanchor": "bottom", "y": -0.22, "x": 0, "font": {"size": 11}},
    )
    figure.update_xaxes(showgrid=False, linecolor=GRID, ticks="outside", tickcolor=GRID)
    figure.update_yaxes(
        title_text=unit, showgrid=True, gridcolor=GRID, zeroline=False, linecolor=GRID
    )
    return figure

## 3. Card 1 — Avondpieken

The highest evening peak per day, with the calmer weekly median over it. Purely descriptive:
no target value and no judging colour, in line with the existing capacity and baseload
analyses. Sketch: figuur A2.

In [6]:
figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=daily["day"],
        y=daily["evening_peak_in_kilowatt"],
        name="Avondpiek per dag",
        mode="lines",
        line={"color": SERIES_FAINT, "width": 1},
        fill="tozeroy",
        fillcolor=SERIES_FILL,
        hovertemplate="%{y:.2f} kW<extra></extra>",
    )
)
figure.add_trace(
    go.Scatter(
        x=week_medians["week"],
        y=week_medians["median_evening_peak_in_kilowatt"],
        name="Weekmediaan",
        mode="lines",
        line={"color": SERIES, "width": 2.5, "shape": "hv"},
        hovertemplate="%{y:.2f} kW<extra></extra>",
    )
)
style(
    figure,
    "Avondpieken",
    "Je hoogste kwartiervermogen tussen 16:00 en 21:00, per dag. De dikke lijn is de mediaan per week.",
    "kW",
)

## 4. Card 2 — Piekaandeel

The same shape, plus a reference line at 37% — the peak share of an average Dutch household.
The reference wears muted ink rather than a series colour, and is labelled directly, because it
is an annotation and not a third measurement. Sketch: figuur A3.

The three-day January gap appears as nulls on the gapless daily index, so both line charts break where the data breaks rather than interpolating across it.

In [7]:
threshold_percent = analysis_input.peak_share_threshold * 100

figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=daily["day"],
        y=daily["evening_peak_share_in_percent"],
        name="Piekaandeel per dag",
        mode="lines",
        line={"color": SERIES_FAINT, "width": 1},
        fill="tozeroy",
        fillcolor=SERIES_FILL,
        hovertemplate="%{y:.1f}%<extra></extra>",
    )
)
figure.add_trace(
    go.Scatter(
        x=week_medians["week"],
        y=week_medians["median_evening_peak_share_in_percent"],
        name="Weekmediaan",
        mode="lines",
        line={"color": SERIES, "width": 2.5, "shape": "hv"},
        hovertemplate="%{y:.1f}%<extra></extra>",
    )
)
# The reference goes in the legend rather than as an inline annotation: at 37% the
# label lands straight on top of the median line. It is an annotation, not a third
# measurement, so it wears muted ink instead of a series colour.
figure.add_trace(
    go.Scatter(
        x=[daily["day"].min(), daily["day"].max()],
        y=[threshold_percent, threshold_percent],
        name=f"Referentie {threshold_percent:.0f}%",
        mode="lines",
        line={"color": INK_MUTED, "width": 1, "dash": "dash"},
        hoverinfo="skip",
    )
)
style(
    figure,
    "Piekaandeel",
    "Het deel van je dagverbruik dat tussen 16:00 en 21:00 valt. Stroom die je teruglevert telt"
    " niet mee als negatief verbruik.<br>De stippellijn op 37% is het piekaandeel van een"
    " gemiddeld huishouden in Nederland.",
    "%",
)
# Bounded to the range the data actually occupies: a 0-100% axis leaves the top half
# empty and flattens the day-to-day variation that is the point of the chart.
figure.update_yaxes(range=[0, 70], ticksuffix="%")
figure.update_layout(margin={"l": 60, "r": 30, "t": 105, "b": 50})
figure

## 5. The key figures

The four numbers shown under each card. `daysBelowThreshold` and `measuredDays` are the
numerator and denominator of the "x of y days" figure; both count fully measured days only, so
the three-day January gap and any partial day at either end drop out.

In [8]:
summary = EveningPeakOutput.from_result(
    result,
    analyzer.peak_moments(net_offtake, num_peaks=analysis_input.num_peak_moments),
    peak_share_threshold=analysis_input.peak_share_threshold,
    reference=analysis_input.reference,
).summary

print("Avondpieken")
print(f"  gemiddelde avondpiek   {summary.average_peak_in_kilowatt:5.2f} kW")
print(f"  laagste avondpiek      {summary.lowest_peak_in_kilowatt:5.2f} kW")
print(f"  hoogste avondpiek      {summary.highest_peak_in_kilowatt:5.2f} kW")
print()
print("Piekaandeel")
print(f"  gemiddeld piekaandeel  {summary.average_share_in_percent:5.1f} %")
print(f"  laagste piekaandeel    {summary.lowest_share_in_percent:5.1f} %")
print(f"  hoogste piekaandeel    {summary.highest_share_in_percent:5.1f} %")
print(
    f"  dagen onder {summary.threshold_in_percent:.0f}%        "
    f"{summary.days_below_threshold} van {summary.measured_days} gemeten dagen"
)
print()
print(f"periode                  {summary.first_day} – {summary.last_day}")
print(
    f"dagen zonder meting      {(summary.last_day - summary.first_day).days + 1 - summary.measured_days}"
)

Avondpieken
  gemiddelde avondpiek    2.78 kW
  laagste avondpiek       0.19 kW
  hoogste avondpiek       5.64 kW

Piekaandeel
  gemiddeld piekaandeel   37.1 %
  laagste piekaandeel      8.2 %
  hoogste piekaandeel     64.7 %
  dagen onder 37%        60 van 117 gemeten dagen

periode                  2026-11-01 – 2027-02-28
dagen zonder meting      3


### How jagged is it?

Worth checking on any sample, real or synthetic: a household's peak share should swing widely.
A narrow band means the data has been smoothed into something that hides exactly the day-to-day
variation the participant is meant to act on.

In [9]:
shares = daily["evening_peak_share_in_percent"].drop_nulls()
peaks = daily["evening_peak_in_kilowatt"].drop_nulls()

print(f"piekaandeel  mediaan {shares.median():.1f}%   spreiding {shares.std():.1f} pp")
print(f"             10e percentiel {shares.quantile(0.1):.1f}%   90e {shares.quantile(0.9):.1f}%")
print(f"avondpiek    mediaan {peaks.median():.2f} kW   {peaks.min():.2f} – {peaks.max():.2f} kW")
print()
print(f"dagen ver onder de drempel (<25%)  {shares.filter(shares < 25).len()}")
print(f"dagen ver boven de drempel (>50%)  {shares.filter(shares > 50).len()}")
print(f"avonden vrijwel niemand thuis (<0.5 kW)  {peaks.filter(peaks < 0.5).len()}")

piekaandeel  mediaan 36.5%   spreiding 11.8 pp
             10e percentiel 25.8%   90e 50.1%
avondpiek    mediaan 2.27 kW   0.19 – 5.64 kW

dagen ver onder de drempel (<25%)  12
dagen ver boven de drempel (>50%)  13
avonden vrijwel niemand thuis (<0.5 kW)  10


## 6. Card 4 — Piekmomenten

The highest evening peaks, each with the curve of its own day. Small multiples rather than one
crowded chart: each panel is a single series, so no colour has to carry identity. The evening
window is shaded and the peak marked, which is what makes the panel readable — you see at a
glance whether the peak is an isolated spike or the top of a long evening. Sketch: figuur A5.

In [10]:
moments = analyzer.peak_moments(net_offtake, num_peaks=6)

figure = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        f"{moment.peak_value_in_kilowatt:.2f} kW · {moment.peak_time:%a %d %b %H:%M}"
        for moment in moments
    ],
    shared_yaxes=True,
    vertical_spacing=0.16,
    horizontal_spacing=0.04,
)

for position, moment in enumerate(moments):
    row, col = divmod(position, 3)
    row, col = row + 1, col + 1
    curve = moment.day_curve
    hours = [timestamp.hour + timestamp.minute / 60 for timestamp in curve["timestamp"]]

    figure.add_vrect(
        x0=analysis_input.window_start.hour,
        x1=analysis_input.window_end.hour,
        fillcolor="rgba(11, 11, 11, 0.05)",
        line_width=0,
        row=row,
        col=col,
    )
    figure.add_trace(
        go.Scatter(
            x=hours,
            y=curve["power_in_kilowatt"],
            mode="lines",
            line={"color": SERIES, "width": 1.5},
            showlegend=False,
            hovertemplate="%{y:.2f} kW om %{x:.2f}h<extra></extra>",
        ),
        row=row,
        col=col,
    )
    figure.add_trace(
        go.Scatter(
            x=[moment.peak_time.hour + moment.peak_time.minute / 60],
            y=[moment.peak_value_in_kilowatt],
            mode="markers",
            marker={"color": SERIES, "size": 9, "line": {"color": SURFACE, "width": 2}},
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=col,
    )

figure.update_layout(
    title={
        "text": "<b>Piekmomenten</b><br><span style='font-size:12px;color:#52514e'>"
        "Een overzicht van je hoogste avondpieken, met het verloop van die dag eromheen."
        " Het grijze vlak is 16:00–21:00.</span>",
        "font": {"size": 16, "color": INK},
        "x": 0,
        "xref": "paper",
    },
    template="simple_white",
    paper_bgcolor=SURFACE,
    plot_bgcolor=SURFACE,
    font={"family": "system-ui, sans-serif", "size": 11, "color": INK_MUTED},
    height=460,
    margin={"l": 50, "r": 30, "t": 110, "b": 45},
)
figure.update_xaxes(
    range=[0, 24],
    tickvals=[0, 6, 12, 18, 24],
    ticktext=["00", "06", "12", "18", "24"],
    showgrid=False,
    linecolor=GRID,
)
figure.update_yaxes(showgrid=True, gridcolor=GRID, zeroline=False, linecolor=GRID)
figure.update_annotations(font={"size": 11, "color": INK})
figure

## 7. A day in detail

The whole point of the two quantities sitting side by side: the day with the highest peak is not
the day with the highest share. A single oven at 19:00 makes a tall peak on an otherwise quiet
day; a low, long evening on a day when nobody used anything else makes a high share with no peak
worth mentioning.

In [11]:
worst_peak = daily.filter(
    pl.col("evening_peak_in_kilowatt") == pl.col("evening_peak_in_kilowatt").max()
).row(0, named=True)
worst_share = daily.filter(
    pl.col("evening_peak_share_in_percent") == pl.col("evening_peak_share_in_percent").max()
).row(0, named=True)

for label, row in (("hoogste avondpiek", worst_peak), ("hoogste piekaandeel", worst_share)):
    print(
        f"{label:22} {row['day']:%a %d %b}"
        f"   piek {row['evening_peak_in_kilowatt']:5.2f} kW"
        f"   aandeel {row['evening_peak_share_in_percent']:5.1f}%"
        f"   dagverbruik {row['daily_offtake_in_kilowatthour']:5.2f} kWh"
    )

print()
print("Dezelfde dag is dus zelden de slechtste op beide grootheden.")

hoogste avondpiek      Fri 06 Nov   piek  5.64 kW   aandeel  56.9%   dagverbruik 18.88 kWh
hoogste piekaandeel    Tue 01 Dec   piek  5.00 kW   aandeel  64.7%   dagverbruik 13.89 kWh

Dezelfde dag is dus zelden de slechtste op beide grootheden.


## 8. Serialize the typed output

The response body the Data Analytics Engine returns. Dumped with `by_alias=True`, so the keys
match the camelCase HTTP shape the front end receives rather than the internal snake_case.

In [12]:
output = EveningPeakOutput.from_result(
    result,
    analyzer.peak_moments(net_offtake, num_peaks=analysis_input.num_peak_moments),
    peak_share_threshold=analysis_input.peak_share_threshold,
    reference=analysis_input.reference,
)

destination = Path("data/evening_peak/evening_peak_output.json")
# Trailing newline so re-running the notebook does not leave the file for
# pre-commit's end-of-file-fixer to change underneath a commit.
destination.write_text(
    output.model_dump_json(by_alias=True, exclude_none=True, indent=1) + "\n",
    encoding="utf-8",
)

print(f"wrote {destination}  ({destination.stat().st_size / 1024:.0f} kB)")
print(f"keys: {sorted(output.model_dump(by_alias=True))}")

wrote data/evening_peak/evening_peak_output.json  (59 kB)
keys: ['dailyPeak', 'dailyShare', 'peakMoments', 'reference', 'summary', 'weekMedianPeak', 'weekMedianShare']
